# AfriQBench Notebook 05 — Mitiq Zero-Noise Extrapolation\n\nThis notebook adds error mitigation as a **separate companion analysis** to the raw TFIM benchmark. We use Mitiq zero-noise extrapolation (ZNE), preserve the intermediate scaled-circuit results, and quantify both accuracy change and execution overhead.\n\nThe target comparison is\n\n$$E_{\\mathrm{noisy}} \\rightarrow E_{\\mathrm{ZNE}} \\rightarrow E_{\\mathrm{exact}}.$$\n\nA key benchmark rule is that mitigation is **not assumed to help**. The notebook explicitly records whether the mitigated estimate is closer to the exact reference.

## 1. Environment\n\nThe current mitigation environment is intentionally separated because current Mitiq 1.1 supports Python 3.11–3.12. Python 3.12 is recommended.\n\n```bash\npython -m pip install -e ".[quantum,mitigation,notebook]"\n```

In [ ]:
from pathlib import Path\nimport sys\n\nrepo_root = Path.cwd()\nif not (repo_root / 'src').exists():\n    repo_root = repo_root.parent\nsys.path.insert(0, str(repo_root / 'src'))\n\nimport json\nimport importlib.metadata as metadata\nimport numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\n\nfrom afriqbench.models.tfim import tfim_hamiltonian\nfrom afriqbench.quantum.cross_backend import estimate_tfim_energy_device_derived_aer\nfrom afriqbench.quantum.mitigation import run_zne_energy_experiment\nfrom afriqbench.quantum.noise import estimate_tfim_energy_noisy_aer\nfrom afriqbench.quantum.qiskit_tfim import bind_ansatz\nfrom afriqbench.reference.exact import ground_state\n\nprint('Mitiq version:', metadata.version('mitiq'))\n

## 2. Fixed TFIM benchmark instance

In [ ]:
n_qubits = 4\nJ = 1.0\nh = 1.0\nshots = 20_000\nseed = 12345\nscale_factors = [1.0, 2.0, 3.0]\n\nbaseline_path = repo_root / 'results' / 'ideal_quantum_n4_h1_baseline.json'\nbaseline = json.loads(baseline_path.read_text(encoding='utf-8'))\nparameters = np.asarray(baseline['ansatz']['parameters'], dtype=float)\ncircuit = bind_ansatz(n_qubits, parameters, reps=2)\n\nH = tfim_hamiltonian(n_qubits, J=J, h=h, periodic=False)\nexact_energy, _ = ground_state(H)\nstatevector_energy = baseline['ideal_statevector']['variational_energy']\n\nprint(f'Exact reference: {exact_energy:.10f}')\nprint(f'Variational statevector: {statevector_energy:.10f}')\n

## 3. Define the noisy scalar-energy executor\n\nMitiq's ZNE interface needs an executor that maps a circuit to a scalar expectation value. AfriQBench wraps the existing finite-shot TFIM energy estimator.

In [ ]:
def controlled_energy_executor(input_circuit):\n    result = estimate_tfim_energy_noisy_aer(\n        input_circuit,\n        n_qubits=n_qubits,\n        J=J,\n        h=h,\n        periodic=False,\n        shots=shots,\n        seed=seed,\n        single_qubit_error=0.001,\n        two_qubit_error=0.01,\n        readout_error=0.01,\n    )\n    return result['energy']\n

## 4. Run transparent two-stage Mitiq ZNE\n\nAfriQBench uses global unitary folding and linear extrapolation at scale factors 1, 2, and 3. The helper retains the scaled energies, fit diagnostics, circuit sizes, and shot overhead.

In [ ]:
zne_result = run_zne_energy_experiment(\n    circuit,\n    controlled_energy_executor,\n    exact_energy=exact_energy,\n    scale_factors=scale_factors,\n    n_qubits=n_qubits,\n    shots_per_term=shots,\n    periodic=False,\n)\n\nprint(f"Unmitigated energy: {zne_result['unmitigated_energy']:.8f}")\nprint(f"ZNE energy:         {zne_result['mitigated_energy']:.8f}")\nprint(f"Exact energy:       {exact_energy:.8f}")\nprint('Mitigation success:', zne_result['quality']['mitigation_success'])\nprint('Improvement factor:', zne_result['quality']['improvement_factor'])\n

## 5. Inspect the noise-scaling data

In [ ]:
scale_df = pd.DataFrame({\n    'scale_factor': zne_result['scale_factors'],\n    'energy': zne_result['scaled_energies'],\n    'depth': [\n        item['depth']\n        for item in zne_result['cost']['scaled_circuit_resources']\n    ],\n    'size': [\n        item['size']\n        for item in zne_result['cost']['scaled_circuit_resources']\n    ],\n    'two_qubit_gates': [\n        item['two_qubit_gate_count']\n        for item in zne_result['cost']['scaled_circuit_resources']\n    ],\n})\nscale_df\n

## 6. Visualize the extrapolation\n\nThe line shown here is a visualization of the same linear model used for the zero-noise estimate.

In [ ]:
fit = np.polyfit(scale_df['scale_factor'], scale_df['energy'], deg=1)\nx_fit = np.linspace(0.0, max(scale_factors), 100)\ny_fit = np.polyval(fit, x_fit)\n\nfig, ax = plt.subplots(figsize=(7, 4.5))\nax.scatter(scale_df['scale_factor'], scale_df['energy'], label='Measured scaled circuits')\nax.plot(x_fit, y_fit, label='Linear extrapolation')\nax.scatter([0.0], [zne_result['mitigated_energy']], marker='x', s=70, label='ZNE estimate')\nax.axhline(exact_energy, linestyle='--', linewidth=1, label='Exact reference')\nax.set_xlabel('Noise scale factor')\nax.set_ylabel('TFIM energy')\nax.set_title('Mitiq ZNE for the AfriQBench TFIM workload')\nax.legend()\nfig.tight_layout()\nplt.show()\n

## 7. Compare raw and mitigated errors

In [ ]:
quality = zne_result['quality']\nerror_df = pd.DataFrame({\n    'estimate': ['Unmitigated', 'Mitiq ZNE'],\n    'absolute_error': [\n        quality['unmitigated_absolute_error'],\n        quality['mitigated_absolute_error'],\n    ],\n})\n\nfig, ax = plt.subplots(figsize=(6.5, 4.5))\nax.bar(error_df['estimate'], error_df['absolute_error'])\nax.set_ylabel('Absolute energy error')\nax.set_title('Does ZNE improve the application-level result?')\nfig.tight_layout()\nplt.show()\n\nerror_df\n

## 8. Quantify mitigation cost\n\nFor the open four-qubit TFIM there are seven Hamiltonian terms. At 20,000 shots per term, one raw energy estimate costs 140,000 shots. Three ZNE scale factors therefore use 420,000 shots, before considering the extra folded gates.

In [ ]:
zne_result['cost']\n

## 9. Optional: ZNE on device-derived local noise\n\nThis experiment can be more computationally expensive. It is disabled by default but uses the same cached fake-backend path introduced in Notebook 04.

In [ ]:
RUN_DEVICE_DERIVED_ZNE = False\n\nif RUN_DEVICE_DERIVED_ZNE:\n    def device_energy_executor(input_circuit):\n        result = estimate_tfim_energy_device_derived_aer(\n            input_circuit,\n            n_qubits=n_qubits,\n            J=J,\n            h=h,\n            shots=shots,\n            seed=seed,\n            device_id='fake_manila',\n            optimization_level=0,\n        )\n        return result['energy']\n\n    device_zne_result = run_zne_energy_experiment(\n        circuit,\n        device_energy_executor,\n        exact_energy=exact_energy,\n        scale_factors=scale_factors,\n        n_qubits=n_qubits,\n        shots_per_term=shots,\n    )\n    print(json.dumps(device_zne_result['quality'], indent=2))\n

## 10. Save a structured mitigation record\n\nThe output captures the full scale-factor data, fit diagnostics, cost, and the explicit success/failure assessment.

In [ ]:
results_dir = repo_root / 'results'\nresults_dir.mkdir(exist_ok=True)\n\ncsv_path = results_dir / 'mitiq_zne_controlled_n4_h1_run.csv'\njson_path = results_dir / 'mitiq_zne_controlled_n4_h1_run.json'\n\nscale_df.to_csv(csv_path, index=False)\njson_path.write_text(\n    json.dumps({\n        'benchmark_id': 'tfim-mitiq-zne-n4-h1-v0',\n        'mitiq_version': metadata.version('mitiq'),\n        'exact_energy': exact_energy,\n        'variational_statevector_energy': statevector_energy,\n        'noise_model': {\n            'single_qubit_error': 0.001,\n            'two_qubit_error': 0.01,\n            'readout_error': 0.01,\n        },\n        'shots_per_term': shots,\n        'seed': seed,\n        'zne': zne_result,\n    }, indent=2),\n    encoding='utf-8',\n)\n\nprint(csv_path)\nprint(json_path)\n

## Benchmark interpretation\n\nZNE is useful only if its accuracy benefit justifies its extra sampling and circuit cost. AfriQBench therefore retains both the unmitigated result and the mitigation result. The raw result remains the canonical hardware/noise benchmark, while Mitiq ZNE is reported as an optional intervention layer.\n\nWith the five notebook layers now present, the next project step is no longer another physics demonstration. It is to package the TFIM workload into a **Metriq-Gym-compatible benchmark/schema proposal** and prepare an upstream contribution.